# Qolda — проверка на RunPod A40

Цель: проверить два ключевых пункта из плана —
1. **Языки** (kk/ru/en) — качество ответов на текстовых промптах.
2. **Зрение/OCR** — читает ли модель реальные изображения документов.

Запускать на поде (Jupyter в RunPod или через `jupyter lab --ip 0.0.0.0 --allow-root`).
Модель и кэш HF кладём в `/workspace`, чтобы пережить рестарт пода (`/` — всего 5 GB overlay, `/workspace` — сетевой volume).

In [ ]:
# --- 0. Проверка GPU ---
# Если torch.cuda.is_available() == False — под запущен без GPU-обвязки,
# нужно переключить/пересоздать под с GPU в консоли RunPod, дальше ехать некуда.
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    raise SystemExit('GPU недоступна в этом контейнере — сначала почини под, дальше нет смысла бежать.')

In [ ]:
# --- 1. Кэш и пути в /workspace (переживает рестарт пода) ---
import os
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'  # быстрее качает, если стоит hf_transfer
os.makedirs('/workspace/hf_cache', exist_ok=True)
os.makedirs('/workspace/test_docs', exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# --- 2. Зависимости ---
# ВАЖНО: НЕ ставить последнюю transformers (-U). Custom code этой модели (InternVLChatModel)
# писался под API поколения 4.5x. Более новый transformers меняет внутренние методы
# (mark_tied_weights_as_initialized и т.п.) и custom code репозитория с ним падает:
#   AttributeError: 'InternVLChatModel' object has no attribute 'all_tied_weights_keys'
# Поэтому фиксируем версию явно, а не берём "любую >=".
%pip install -q "transformers==4.55.0" accelerate einops timm pillow sentencepiece hf_transfer

print('Если это первая установка в сессии ИЛИ transformers уже был импортирован раньше —')
print('перезапусти ядро (Kernel -> Restart) перед следующей ячейкой, иначе останется старая версия в памяти.')

In [ ]:
# Проверка версии ПОСЛЕ рестарта ядра — если тут не 4.55.0, ядро не перезапущено.
import transformers
print('transformers version:', transformers.__version__)
assert transformers.__version__ == '4.55.0', 'Перезапусти ядро — установка не подхватилась.'

## Загрузка модели

Идём через `transformers` напрямую (не LMDeploy-сервер) — так проще отлаживать построчно в ноутбуке.
Если этот путь заведётся, потом эквивалентно поднимем `lmdeploy serve api_server` для реального сервиса.

`issai/Qolda` — кастомная архитектура `InternVLChatModel`, поэтому `trust_remote_code=True` обязателен.

In [ ]:
import time
from transformers import AutoModel, AutoTokenizer

MODEL_ID = 'issai/Qolda'

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).eval().cuda()
print(f'Загружено за {time.time()-t0:.1f}s')
print('VRAM занято, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

## Тест 1 — языки (kk/ru/en), без изображений

In [ ]:
import re

def split_thinking(raw_answer):
    """model.chat() у этой custom-архитектуры не принимает enable_thinking как kwarg —
    это переменная chat_template.jinja, применяемая только внутри apply_chat_template()
    (используется LMDeploy-сервером). Прямой .chat() всегда думает и отдаёт <think>...</think>
    перед финальным ответом. Разбираем это здесь руками."""
    m = re.search(r'<think>(.*?)</think>\s*(.*)', raw_answer, flags=re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return None, raw_answer.strip()

# pad_token_id указан явно, чтобы не сыпалось предупреждение
# "Setting `pad_token_id` to `eos_token_id`" при каждой генерации.
GEN_KW = dict(max_new_tokens=768, do_sample=False, pad_token_id=tokenizer.eos_token_id)

def ask_text(question):
    raw = model.chat(tokenizer, None, question, GEN_KW)
    return split_thinking(raw)

prompts = {
    'kk': 'Қазақстанның бас қаласы қандай және неге астана болып таңдалды? Қысқаша жауап бер.',
    'ru': 'Объясни своими словами, чем отличается ИП от ТОО в Казахстане. Кратко.',
    'en': 'In two sentences, explain what a vector database is used for in RAG systems.',
}

for lang, q in prompts.items():
    t0 = time.time()
    thinking, answer = ask_text(q)
    dt = time.time() - t0
    print(f'=== {lang} ({dt:.1f}s) ===')
    print('Q:', q)
    if thinking:
        print('[thinking]:', thinking[:300], '...' if len(thinking) > 300 else '')
    print('A:', answer)
    print()

**На что смотреть:** не съезжает ли модель на другой язык посреди ответа, держит ли терминологию, адекватна ли грамматика на kk — это самое слабое место у большинства open-source LLM.

## Тест 2 — зрение / OCR документа

Положи тестовый скан (фото/PDF-страницу как PNG/JPG) в `/workspace/test_docs/sample.png` перед запуском этой ячейки.
Пока файла нет — ячейка ниже создаёт синтетическое изображение с текстом kk/ru, чтобы проверить пайплайн сквозным образом.

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

SAMPLE_PATH = Path('/workspace/test_docs/sample.png')

if not SAMPLE_PATH.exists():
    print('Реального скана нет — генерирую синтетический тестовый документ.')
    img = Image.new('RGB', (1000, 500), 'white')
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 28)
    except Exception:
        font = ImageFont.load_default()
    lines = [
        'АНЫҚТАМА / СПРАВКА',
        'Тегі: Ахметов А.Б.',
        'Мекенжайы: Алматы қаласы, Абай көшесі 10',
        'Дата выдачи: 06.08.2026',
        'Номер документа: 45231-KZ',
    ]
    for i, line in enumerate(lines):
        draw.text((40, 40 + i * 60), line, fill='black', font=font)
    img.save(SAMPLE_PATH)
    print('Синтетический документ сохранён:', SAMPLE_PATH)
else:
    print('Использую реальный файл:', SAMPLE_PATH)

img = Image.open(SAMPLE_PATH)
print('Размер:', img.size)
img

In [ ]:
# --- Препроцессинг под InternVL: тайлы 448x448 ---
# Стандартный InternVL image-transform (взято из практики репозиториев InternVL/Qolda).
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda im: im.convert('RGB') if im.mode != 'RGB' else im),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):
    """max_num ограничивает число тайлов — держим маленьким, чтобы не раздувать контекст на A4."""
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    target_ratios = sorted(
        {(i, j) for n in range(min_num, max_num + 1)
         for i in range(1, n + 1) for j in range(1, n + 1)
         if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1],
    )
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    resized_img = image.resize((target_width, target_height))
    processed_images = []
    cols = target_width // image_size
    for i in range(blocks):
        box = (
            (i % cols) * image_size,
            (i // cols) * image_size,
            (i % cols + 1) * image_size,
            (i // cols + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and blocks != 1:
        processed_images.append(image.resize((image_size, image_size)))
    return processed_images

def load_image_as_pixel_values(image_path, input_size=448, max_num=6):
    image = Image.open(image_path).convert('RGB')
    transform = build_transform(input_size=input_size)
    tiles = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(tile) for tile in tiles]
    pixel_values = torch.stack(pixel_values)
    print(f'Тайлов: {len(tiles)} (max_num={max_num})')
    return pixel_values

In [ ]:
pixel_values = load_image_as_pixel_values(SAMPLE_PATH, max_num=6).to(torch.bfloat16).cuda()

question_vision = '<image>\nПрочитай весь текст на изображении дословно, ничего не придумывай. Если что-то нечитаемо — так и скажи.'

t0 = time.time()
raw = model.chat(tokenizer, pixel_values, question_vision, GEN_KW)
dt = time.time() - t0
thinking, answer = split_thinking(raw)
print(f'--- Ответ ({dt:.1f}s) ---')
if thinking:
    print('[thinking]:', thinking[:300], '...' if len(thinking) > 300 else '')
print(answer)

**Критерий прохождения (шаг 0 из плана):** ответ содержит реальный текст с картинки (для синтетики — строки про АНЫҚТАМА/СПРАВКА, ФИО, адрес, дату, номер), а не галлюцинацию не по теме.

Если ответ бессвязный или не относится к содержимому — воспроизводится известный баг InternVL3.5 с некорректным чтением текста на изображениях. В этом случае план предусматривает переход на LMDeploy (`--backend pytorch`, как рекомендует карточка модели) или на связку внешний OCR + текстовая модель.

*Примечание про `enable_thinking`:* в README он упоминается как параметр `extra_body` для LMDeploy-сервера (`apply_chat_template` берёт его из jinja-шаблона). Прямой вызов `model.chat()` из `modeling_internvl_chat.py` такого параметра не принимает и всегда генерирует `<think>...</think>` перед ответом — поэтому здесь мы просто парсим и отделяем этот блок функцией `split_thinking()`, а не пытаемся его отключить.

In [ ]:
# --- Доп. вопрос по документу (проверка понимания, а не только распознавания) ---
question_followup = '<image>\nКакая дата указана в документе и какой у него номер?'
raw2 = model.chat(tokenizer, pixel_values, question_followup, GEN_KW)
_, answer2 = split_thinking(raw2)
print(answer2)

## Замер VRAM и скорости

In [ ]:
print('VRAM allocated, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))
print('VRAM reserved,  GB:', round(torch.cuda.memory_reserved() / 2**30, 2))
print('VRAM peak,      GB:', round(torch.cuda.max_memory_allocated() / 2**30, 2))

## Дальше по плану

1. Если зрение прошло — положить реальные (обезличенные) сканы в `/workspace/test_docs/` и повторить тест 2 на них.
2. Поднять `lmdeploy serve api_server issai/Qolda --backend pytorch` как постоянный сервис — это то, к чему будет ходить `app/backends/openai_compat.py` из каркаса.
3. Повторить оба теста на `Sherkala-8B` и `KazLLM-8B` (текстовая часть — они зрения не умеют) для сравнения качества kk/ru.
4. Сравнить bf16 (этот ноутбук) с Q4_K_M GGUF — ответ на вопрос, хватит ли P2000 локально.